# GSA workflow of Your Project

Based on Weier Liu's code

In [ ]:
import pandas as pd
import bw2data as bd
import bw2io as bi

In [ ]:
%load_ext autoreload
%autoreload 2

from utils import lca_project_setup
from utils import lca_parameters
from utils import lca_contribution
from utils import lca_monte_carlo
from utils import lca_sobol

## 0. Brightway setup

In [ ]:
# bd calles the bw2data functions, the .projects part asks bw2data to call all the projects
bd.projects

In [ ]:
bd.projects.set_current('your_project_name') # insert the name of your project 
bd.databases

### Import ecoinvent and biosphere

In [ ]:
del bd.databases['your_ecoinvent_db-consequential'] # delete the biosphere database, if it exists

In [ ]:
if 'your_ecoinvent_db-consequential' in bd.databases:
    print('your_ecoinvent_db-consequential already exists')
else:
    bi.import_ecoinvent_release(
    version='3.11',
    system_model='consequential',
    username = "zhengxiaoyu@xhlab.ac.cn",   # your username
    password= "Xhsys@2022"   # your password
)
    


### Import the custom databases

In [ ]:
# Import custom database: your_auxiliary_db_1
path1 = "./data/your_auxiliary_db_1.xlsx"
if "your_auxiliary_db_1" in bd.databases:
    print("your_auxiliary_db_1 database already imported.")
else:
    ei = bi.ExcelImporter(path1) 
    ei.apply_strategies()
    ei.match_database('your_ecoinvent_db-biosphere') 
    ei.match_database('your_ecoinvent_db-consequential') 
    ei.statistics() # Get the overview of our LCI in Brightway. Check if there are unlinked exchanges
    ei.write_project_parameters() 
    ei.write_database(activate_parameters=True)

# Import custom database: your_auxiliary_db_2
path2 = "./data/your_auxiliary_data.xlsx"
if "your_auxiliary_db_2" in bd.databases:
    print("your_auxiliary_db_2 database already imported.")
else:
    ei = bi.ExcelImporter(path2) 
    ei.apply_strategies()
    ei.match_database('your_ecoinvent_db-biosphere') 
    ei.match_database('your_ecoinvent_db-consequential') 
    ei.statistics() # Get the overview of our LCI in Brightway. Check if there are unlinked exchanges
    ei.write_project_parameters() 
    ei.write_database(activate_parameters=True)

# Import custom database: Foreground processes (your_foreground_db)
path = "./Your Project cases/Pyro_Phenol_v16_CHP.xlsx"
if "your_foreground_db" in bd.databases:
    print("your_foreground_db database already imported.")
else:
    ei = bi.ExcelImporter(path) 
    ei.apply_strategies()
    ei.match_database('your_ecoinvent_db-biosphere') 
    ei.match_database('your_ecoinvent_db-consequential')
    ei.match_database('your_auxiliary_db_1')
    ei.match_database('your_auxiliary_db_2')
    ei.statistics() # Get the overview of our LCI in Brightway. Check if there are unlinked exchanges
    ei.write_project_parameters() 
    ei.write_database(activate_parameters=True)

## 1. Project configuration

In [ ]:
# select LCIA method
for m in bd.methods:
    #if 'cliamte change' in m[2] and 'GWP100' in m[3] :
    if 'climate change' in m[2]:
        print(m)

In [ ]:
PROJECT_NAME = 'your_project_name'
FOREGROUND_DB = 'your_foreground_db'
GROUP_NAME = 'your_parameter_group'

cc_method = ('your_ecoinvent_db', 'EF v3.1', 'climate change', 'global warming potential (GWP100)')
pm_method = ('your_ecoinvent_db', 'EF v3.1', 'particulate matter formation', 'impact on human health')
eu_method_fw = ('your_ecoinvent_db', 'EF v3.1', 'eutrophication: freshwater', 'fraction of nutrients reaching freshwater end compartment (P)')
eu_method_mr = ('your_ecoinvent_db', 'EF v3.1', 'eutrophication: marine', 'fraction of nutrients reaching marine end compartment (N)')
METHODS = [cc_method, pm_method, eu_method_fw, eu_method_mr]

lca_project_setup.set_project(PROJECT_NAME)
list(bd.databases)

## 2. Select functional units

Prefer selecting by code once and storing them in a list.

In [ ]:
for a in bd.Database('your_foreground_db'): # notice to call a Database you must capitalize the D and insert the right name in parenthesis 
    print(a['name'],a['code'])

In [ ]:
FU_CODES = [
    'your_fu_code'
]
FUS = [bd.Database(FOREGROUND_DB).get(code) for code in FU_CODES]
[(fu['name'], fu['code']) for fu in FUS]

## 3. Register exchanges and validate formulas

Run this after importing/updating the foreground database.

In [ ]:
lca_parameters.add_database_exchanges_to_group(FOREGROUND_DB, GROUP_NAME)
formula_check_df = lca_parameters.validate_exchange_formulas(FOREGROUND_DB, GROUP_NAME)
formula_check_df.to_excel('formula_check_results.xlsx', index=False)
formula_check_df.head()

## 4. Deterministic LCA

In [ ]:
deterministic_df = lca_project_setup.deterministic_lca(FUS, METHODS)
deterministic_df.to_excel('deterministic_results.xlsx', index=False)
deterministic_df

## 5. Contribution analysis

In [ ]:
contribution_df = lca_contribution.contribution_analysis(FUS, METHODS, max_level=1, cutoff=0.001)
contribution_df.head()

In [ ]:
contribution_df.to_excel('contribution_analysis_t2_results.xlsx', index=False)

## 6. OAT sensitivity

In [ ]:
sensitivity_df = lca_contribution.oat_sensitivity(FUS, METHODS, GROUP_NAME, rel_step=0.1)
sensitivity_df.to_excel('oat_sensitivity_results.xlsx', index=False)
sensitivity_df.head()

## 7. Analytical variance

In [ ]:
param_meta_df = lca_contribution.parameter_metadata(GROUP_NAME)
analytical_df = lca_contribution.combine_oat_and_analytical_variance(sensitivity_df, param_meta_df)
analytical_df.to_excel('analytical_sensitivity_results.xlsx', index=False)
analytical_df.head()

## 8. Parallel Monte Carlo

In [ ]:
import multiprocessing
cpu_count = multiprocessing.cpu_count()
print(f"cpu_count: {cpu_count}")

In [ ]:
mc_df = lca_monte_carlo.run_parallel_monte_carlo(
    project_name=PROJECT_NAME,
    group_name=GROUP_NAME,
    functional_units=FUS,
    methods=METHODS,
    n_samples=1000,
    n_workers=6,
    random_seed=42,
    export_parameter_samples='parameter_samples.xlsx',
)
mc_df.to_excel('monte_carlo_results.xlsx', index=False)
mc_df.head()

### Visualization

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde

# MC results
df = pd.read_excel("monte_carlo_results.xlsx").copy()

# deterministic_df is assumed to already exist in memory
det_df = deterministic_df.copy()

# Keep only columns we need and align names
det_df = det_df.rename(columns={"Score": "Deterministic Score"})

# Clean and order categories/scenarios
impact_categories = list(df["Method"].dropna().unique())
scenarios = list(df["Functional unit"].dropna().unique())

# Optional nicer labels for legends
scenario_labels = {
    "0 conventional__0": "0 Conventional System",
    "1 displace conventional__1": "1 Displace Conventional Production",
    "2 displace vegetable__2": "2 Displace Vegetable Production",
    "3 displace vege protein__3": "3 Displace Vegetal Protein",
    "4 displace meat__4": "4 Displace Meat",
}

summary_rows = []

for impact in impact_categories:
    subset = df[df["Method"] == impact].copy()
    all_values = subset["LCA Score"].astype(float).to_numpy()
    if len(all_values) == 0:
        continue

    x_min, x_max = np.nanmin(all_values), np.nanmax(all_values)
    x_pad = (x_max - x_min) * 0.08 if x_max != x_min else max(abs(x_max) * 0.08, 1)
    x_grid = np.linspace(x_min - x_pad, x_max + x_pad, 500)

    plt.figure(figsize=(10, 6))

    for scenario in scenarios:
        vals = subset.loc[
            subset["Functional unit"] == scenario, "LCA Score"
        ].dropna().astype(float).to_numpy()

        if len(vals) < 2:
            continue

        kde = gaussian_kde(vals)
        y = kde(x_grid)
        label = scenario_labels.get(scenario, scenario)

        plt.plot(x_grid, y, label=label, linewidth=2)

        # deterministic point for this scenario + impact
        det_match = det_df.loc[
            (det_df["Functional unit"] == scenario) &
            (det_df["Method"] == impact),
            "Deterministic Score"
        ]

        if not det_match.empty:
            x_det = float(det_match.iloc[0])
            y_det = float(kde(x_det))
            plt.scatter(
                [x_det],
                [y_det],
                s=60,
                zorder=5,
            )

        summary_rows.append({
            "Impact category": impact,
            "Scenario": scenario,
            "n": len(vals),
            "Mean": float(np.mean(vals)),
            "Median": float(np.median(vals)),
            "SD": float(np.std(vals, ddof=1)),
            "Min": float(np.min(vals)),
            "Max": float(np.max(vals)),
            "Deterministic Score": float(det_match.iloc[0]) if not det_match.empty else np.nan,
        })

    plt.xlabel("LCA Score")
    plt.ylabel("Probability density")
    plt.title(f"Monte Carlo distribution: {impact}")
    plt.legend(frameon=False)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()

    print(f"Generated plot for impact category: {impact}")
    plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde
import math
import re
import textwrap

# MC results
df = pd.read_excel("monte_carlo_results.xlsx").copy()

# deterministic_df is assumed to already exist in memory
det_df = deterministic_df.copy()

# Keep only columns we need and align names
det_df = det_df.rename(columns={"Score": "Deterministic Score"})

# Clean and order categories/scenarios
impact_categories = list(df["Method"].dropna().unique())
scenarios = list(df["Functional unit"].dropna().unique())

# Optional nicer labels for legends
scenario_labels = {
    "0 conventional__0": "0 Conventional System",
    "1 displace conventional__1": "1 Displace Conventional Production",
    "2 displace vegetable__2": "2 Displace Vegetable Production",
    "3 displace vege protein__3": "3 Displace Vegetal Protein",
    "4 displace meat__4": "4 Displace Meat",
}

def normalize_name(x: str) -> str:
    s = str(x).strip().lower()
    s = re.sub(r"_+", "_", s)
    s = re.sub(r"\s+", " ", s)
    return s

def pretty_scenario_label(raw_name: str, label_map: dict) -> str:
    raw_norm = normalize_name(raw_name)
    normalized_map = {normalize_name(k): v for k, v in label_map.items()}
    return normalized_map.get(raw_norm, raw_name)

summary_rows = []

# Fixed 2x2 panel
n_panels = len(impact_categories)
ncols = 2
nrows = 2

fig, axes = plt.subplots(nrows, ncols, figsize=(14, 10), squeeze=False)
axes_flat = axes.flatten()

for idx, impact in enumerate(impact_categories[:4]):
    ax = axes_flat[idx]

    subset = df[df["Method"] == impact].copy()
    all_values = subset["LCA Score"].astype(float).to_numpy()

    if len(all_values) == 0:
        ax.set_visible(False)
        continue

    x_min, x_max = np.nanmin(all_values), np.nanmax(all_values)
    x_pad = (x_max - x_min) * 0.08 if x_max != x_min else max(abs(x_max) * 0.08, 1)
    x_grid = np.linspace(x_min - x_pad, x_max + x_pad, 500)

    for scenario in scenarios:
        vals = subset.loc[
            subset["Functional unit"] == scenario, "LCA Score"
        ].dropna().astype(float).to_numpy()

        if len(vals) < 2:
            continue

        kde = gaussian_kde(vals)
        y = kde(x_grid)
        label = pretty_scenario_label(scenario, scenario_labels)

        line, = ax.plot(x_grid, y, label=label, linewidth=2)

        # deterministic point for this scenario + impact
        det_match = det_df.loc[
            (det_df["Functional unit"] == scenario) &
            (det_df["Method"] == impact),
            "Deterministic Score"
        ]

        if not det_match.empty:
            x_det = float(det_match.iloc[0])
            y_det = float(kde(x_det))
            ax.scatter(
                [x_det],
                [y_det],
                s=50,
                zorder=5,
                color=line.get_color(),
                edgecolors="black",
                linewidths=0.6,
            )

        summary_rows.append({
            "Impact category": impact,
            "Scenario": scenario,
            "n": len(vals),
            "Mean": float(np.mean(vals)),
            "Median": float(np.median(vals)),
            "SD": float(np.std(vals, ddof=1)),
            "Min": float(np.min(vals)),
            "Max": float(np.max(vals)),
            "Deterministic Score": float(det_match.iloc[0]) if not det_match.empty else np.nan,
        })

    ax.set_xlabel("LCA Score")
    ax.set_ylabel("Probability density")
    ax.set_title("\n".join(textwrap.wrap(f"{impact}", width=45)))
    ax.grid(True, alpha=0.3)

# Hide unused panels if fewer than 4 methods
for j in range(min(4, n_panels), 4):
    axes_flat[j].set_visible(False)

# One shared legend for the whole figure
handles, labels = axes_flat[0].get_legend_handles_labels()
fig.legend(
    handles,
    labels,
    loc="lower center",
    bbox_to_anchor=(0.5, 0.01),
    ncol=2,
    frameon=False,
)

fig.tight_layout(rect=[0, 0.06, 1, 1])
plt.show()

summary_df = pd.DataFrame(summary_rows)

## 9. Parallel GSA

In [ ]:
import importlib
from utils import lca_project_setup
from utils import lca_parameters
from utils import lca_contribution
from utils import lca_monte_carlo
from utils import lca_sobol
for mod in [lca_project_setup, lca_parameters, lca_contribution, lca_monte_carlo, lca_sobol]:
    importlib.reload(mod)

In [ ]:
sobol_problem, sobol_plist = lca_sobol.build_sobol_problem('your_parameter_group')

sobol_samples = lca_sobol.generate_sobol_samples(
    sobol_problem,
    N=512,
    calc_second_order=False,
    scramble=True,
    seed=42,
)

sobol_samples.shape

In [ ]:
sobol_df = lca_sobol.run_parallel_sobol_from_samples(
    project_name=PROJECT_NAME,
    group_name=GROUP_NAME,
    functional_units=FUS,
    methods=METHODS,
    sobol_problem=sobol_problem,
    sobol_samples=sobol_samples,
    n_workers=6,
    verbose=True,
)

In [ ]:
sobol_df.to_excel('sobol_samples_results.xlsx', index=False)

In [ ]:
all_sobol_tables = []

pairs = (
    sobol_df[["Functional unit", "Method full"]]
    .drop_duplicates()
    .itertuples(index=False, name=None)
)

for fu_label, method_tuple in pairs:
    Si = lca_sobol.sobol_indices_from_results(
        sobol_problem=sobol_problem,
        sobol_results_df=sobol_df,
        functional_unit=fu_label,
        method=method_tuple,
        calc_second_order=False,
    )

    tmp = lca_sobol.sobol_indices_to_dataframe(Si, sobol_problem)
    tmp["Functional unit"] = fu_label
    tmp["Method full"] = [method_tuple] * len(tmp)
    tmp["Method"] = (
        method_tuple[2]
        if isinstance(method_tuple, tuple) and len(method_tuple) > 2
        else str(method_tuple)
    )

    all_sobol_tables.append(tmp)

sobol_indices_df = pd.concat(all_sobol_tables, ignore_index=True)

# Optional: reorder columns
sobol_indices_df = sobol_indices_df[
    [
        "Functional unit",
        "Method",
        "Method full",
        "Parameter",
        "S1",
        "S1_conf",
        "ST",
        "ST_conf",
    ]
]

sobol_indices_df.to_excel('sobol_indices_results.xlsx', index=False)
sobol_indices_df.head()

### Visualization

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import math
import textwrap
import re

# -------------------------------------------------
# Inputs
# -------------------------------------------------
df = sobol_indices_df.copy()

scenario_labels = {
    "0 conventional__0": "0 Conventional System",
    "1 displace conventional__1": "1 Displace Conventional Production",
    "2 displace vegetable__2": "2 Displace Vegetable Production",
    "3 displace vege protein__3": "3 Displace Vegetal Protein",
    "4 displace meat__4": "4 Displace Meat",
}

save_path = None   # e.g. "sobol_by_method_panel.pdf"
ncols = 1
top_k = 6

# -------------------------------------------------
# Clean Sobol components
# -------------------------------------------------
df["S1_clean"] = df["S1"].clip(lower=0)
df["interaction_clean"] = (df["ST"] - df["S1"]).clip(lower=0)
df["ST_clean"] = df["S1_clean"] + df["interaction_clean"]

# -------------------------------------------------
# Helpers
# -------------------------------------------------
def normalize_fu_name(x: str) -> str:
    s = str(x).strip().lower()
    s = re.sub(r"_+", "_", s)
    s = re.sub(r"\s+", " ", s)
    return s

def map_fu_label(fu_raw: str, label_map: dict) -> str:
    fu_norm = normalize_fu_name(fu_raw)

    normalized_map = {
        normalize_fu_name(k): v
        for k, v in label_map.items()
    }

    if fu_norm in normalized_map:
        return normalized_map[fu_norm]

    for k_norm, v in normalized_map.items():
        if k_norm in fu_norm or fu_norm in k_norm:
            return v

    return str(fu_raw)

# -------------------------------------------------
# Order FUs using scenario label order when possible
# -------------------------------------------------
label_order = [normalize_fu_name(k) for k in scenario_labels.keys()]
fu_candidates = list(df["Functional unit"].dropna().unique())

fus_ranked = []
for fu in fu_candidates:
    fu_norm = normalize_fu_name(fu)
    try:
        rank = label_order.index(fu_norm)
    except ValueError:
        rank = 999
    fus_ranked.append((rank, str(fu), fu))

fus = [fu for _, _, fu in sorted(fus_ranked, key=lambda x: (x[0], x[1]))]

# Keep method order as it appears
methods = list(df["Method"].dropna().unique())

# -------------------------------------------------
# Global parameter order: largest to smallest
# -------------------------------------------------
global_param_order_df = (
    df.groupby("Parameter", as_index=False)["ST_clean"]
    .mean()
    .sort_values(["ST_clean", "Parameter"], ascending=[False, True])
    .head(top_k)
)

global_param_order = list(global_param_order_df["Parameter"])

# -------------------------------------------------
# Figure layout
# -------------------------------------------------
n_methods = len(methods)
nrows = math.ceil(n_methods / ncols)

fig, axes = plt.subplots(
    nrows,
    ncols,
    figsize=(14, 2.6 * nrows),
    squeeze=False,
    sharey=True,
)

for i, method in enumerate(methods):
    ax = axes[i // ncols][i % ncols]

    sub_method = df[df["Method"] == method].copy()

    group_gap = 0.9
    bar_width = 0.82

    x_positions = []
    x_labels = []
    centers = []
    boundary_positions = []
    plot_rows = []

    x_cursor = 0.0

    for fu_idx, fu in enumerate(fus):
        sub_fu = sub_method[sub_method["Functional unit"] == fu].copy()
        sub_fu = sub_fu.set_index("Parameter")

        sub_fu = sub_fu.reindex(global_param_order).fillna(0)

        sub_fu = sub_fu.sort_values("ST_clean", ascending=False)

        param_order_fu = list(sub_fu.index)

        group_x = []

        for p in param_order_fu:
            row = sub_fu.loc[p]

            plot_rows.append({
            "x": x_cursor,
                "Parameter": p,
                "Functional unit": fu,
                "S1_clean": float(row["S1_clean"]),
                "interaction_clean": float(row["interaction_clean"]),
                "ST_clean": float(row["ST_clean"]),
            })

            x_positions.append(x_cursor)
            x_labels.append(p)
            group_x.append(x_cursor)
            x_cursor += 1.0

        centers.append(np.mean(group_x))

        if fu_idx < len(fus) - 1:
            boundary_positions.append(x_cursor - 0.5 + group_gap / 2.0)
            x_cursor += group_gap

    plot_df = pd.DataFrame(plot_rows)

    ax.bar(
        plot_df["x"],
        plot_df["S1_clean"],
        width=bar_width,
        label="Main effect (S1)",
        linewidth=0,
    )

    ax.bar(
        plot_df["x"],
        plot_df["interaction_clean"],
        bottom=plot_df["S1_clean"],
        width=bar_width,
        label="Interaction/nonlinearity (ST - S1)",
        linewidth=0,
    )

    ax.set_xticks(x_positions)
    ax.set_xticklabels(x_labels, rotation=90, fontsize=8)

    for xpos in boundary_positions:
        ax.axvline(x=xpos, linewidth=1.1, alpha=0.8)

    # FU labels above each group
    trans = ax.get_xaxis_transform()
    for fu, cx in zip(fus, centers):
        fu_label = map_fu_label(fu, scenario_labels)
        fu_label_wrapped = "\n".join(textwrap.wrap(fu_label, width=18))
        ax.text(
            cx,
            1.01,
            fu_label_wrapped,
            transform=trans,
            ha="center",
            va="bottom",
            fontsize=8,
            fontweight="bold",
            linespacing=0.95,
        )

    method_label = "\n".join(textwrap.wrap(str(method), width=55))
    ax.set_title(method_label, fontsize=12, pad=32)

    ax.set_ylim(0, 1.0)
    ax.grid(axis="y", alpha=0.25, linewidth=0.6)
    ax.set_axisbelow(True)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    if len(x_positions) > 0:
        ax.set_xlim(min(x_positions) - bar_width / 2, max(x_positions) + bar_width / 2)

    if i % ncols == 0:
        ax.set_ylabel("Sobol index", fontsize=10)

# Remove unused axes
for j in range(n_methods, nrows * ncols):
    fig.delaxes(axes[j // ncols][j % ncols])

handles, labels = axes[0][0].get_legend_handles_labels()
fig.legend(
    handles,
    labels,
    loc="lower center",
    bbox_to_anchor=(0.5, 0.01),
    ncol=2,
    frameon=False,
    fontsize=10,
)

fig.tight_layout(rect=[0, 0.06, 1, 0.95])

if save_path:
    fig.savefig(save_path, dpi=300, bbox_inches="tight")

plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import math

df = sobol_indices_df.copy()

# Apply clean values
df["S1_clean"] = df["S1"].clip(lower=0)
df["interaction"] = (df["ST"] - df["S1"]).clip(lower=0)

# Your FU labels
scenario_labels = {
    "0 conventional__0": "0 Conventional System",
    "1 displace conventional__1": "1 Displace Conventional Production",
    "2 displace vegetable__2": "2 Displace Vegetable Production",
    "3 displace vege protein__3": "3 Displace Vegetal Protein",
    "4 displace meat__4": "4 Displace Meat",
}

# Unique combinations
pairs = (
    df[["Functional unit", "Method"]]
    .drop_duplicates()
    .itertuples(index=False, name=None)
)
pairs = list(pairs)

# Layout
n = len(pairs)
ncols = 2
nrows = math.ceil(n / ncols)

fig, axes = plt.subplots(nrows, ncols, figsize=(14, 4 * nrows), squeeze=False)

for idx, (fu, method) in enumerate(pairs):
    ax = axes[idx // ncols][idx % ncols]

    sub = df[
        (df["Functional unit"] == fu) &
        (df["Method"] == method)
    ].copy()

    sub = sub.sort_values("ST", ascending=False)

    x = np.arange(len(sub))

    bars1 = ax.bar(
        x,
        sub["S1_clean"],
        label="S1 (main effect)",
    )

    bars2 = ax.bar(
        x,
        sub["interaction"],
        bottom=sub["S1_clean"],
        label="ST - S1 (interaction)",
    )

    ax.set_xticks(x)
    ax.set_xticklabels(sub["Parameter"], rotation=45, ha="right")

    # Apply nicer FU label
    fu_label = scenario_labels.get(fu, fu)

    ax.set_title(f"{fu_label}\n{method}")
    ax.set_ylabel("Sobol index")
    ax.set_ylim(0, 1)
    ax.grid(True, axis="y", alpha=0.3)

# Remove empty axes
for j in range(idx + 1, nrows * ncols):
    fig.delaxes(axes[j // ncols][j % ncols])

# Move legend BELOW figure
handles, labels = ax.get_legend_handles_labels()
fig.legend(
    handles,
    labels,
    loc="lower center",
    ncol=2,
    frameon=False,
)

# Add space at bottom for legend
plt.tight_layout(rect=[0, 0.08, 1, 1])

plt.show()

# **DEBUG**